## Setup runner & utilities

In [ ]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/nanotube.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXPERIMENT: path follower")
imd_runner.load(0)

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [ ]:
TRAIL_COLOR = [0.0, 1.0, 1.0, 1.0]
PATH_COLOR = [1.0, 1.0, 0.0, 1.0]
HOVER_COLOR = [1.0, 0.0, 0.0, 1.0]
CHOSEN_COLOR = [0.0, 1.0, 0.0, 1.0]

In [ ]:
from follower import PathFollowerAgent, Path

PATHS: dict[str, Path] = {}
NEXT_PATH_ID = 0
FOLLOWER_AGENT: PathFollowerAgent | None = None


def CREATE_PATH():
    global NEXT_PATH_ID
    path_id = str(NEXT_PATH_ID)
    NEXT_PATH_ID += 1
    PATHS[path_id] = Path.empty()
    return path_id


def REDRAW_PATH(path_id: str):
    path = PATHS[path_id]
    PATH_OBJECTS.update_line(f"path.{path_id}", positions=path.positions, size=0.05, color=PATH_COLOR)


def STOP_FOLLOWER():
    global FOLLOWER_AGENT
    if FOLLOWER_AGENT is not None:
        FOLLOWER_AGENT.close()
    FOLLOWER_AGENT = None


def START_FOLLOWER(particles: list[int], path: Path):
    global FOLLOWER_AGENT
    STOP_FOLLOWER()
    FOLLOWER_AGENT = PathFollowerAgent.from_runner(imd_runner)
    FOLLOWER_AGENT.set_path(path)
    FOLLOWER_AGENT.set_particles(particles)
    FOLLOWER_AGENT.start()

In [ ]:
from nanover.jupyter import Mode, SceneObjectsUtility
from follower import Path

PATH_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)
CURSOR_DRAWING_PATH_ID = {}


class DrawPathMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            CURSOR_DRAWING_PATH_ID[key] = CREATE_PATH()

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            CURSOR_DRAWING_PATH_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        if not key in CURSOR_DRAWING_PATH_ID:
            return

        path_id = CURSOR_DRAWING_PATH_ID[key]

        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        PATHS[path_id].append_position(next_pos)

        REDRAW_PATH(path_id)


utilities.use_interaction_modes()
utilities.add_interaction_mode(DrawPathMode, "draw path")

In [ ]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode

TRAIL_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)
TRAIL_POINTS = {}


def get_interaction_centroid(interaction: ParticleInteraction):
    indexes = [int(index) for index in interaction.particles]
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    centroid = [float(x) for x in np.mean(positions[indexes], axis=0)]
    return centroid


class InteractionTrailMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        TRAIL_POINTS[key] = []

    def on_interaction_updated(self, *, key: str, interaction: ParticleInteraction):
        TRAIL_POINTS[key].append(get_interaction_centroid(interaction))
        TRAIL_OBJECTS.update_line(f"trail.{key}", positions=TRAIL_POINTS[key], size=0.05, color=TRAIL_COLOR)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        utilities.notify_all(f"FINISHED TRAIL\n{key}")

utilities.add_interaction_mode(InteractionTrailMode, "trails")

In [ ]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from intersection import intersects_sphere_line


SELECTION_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)


def intersect_paths(position, radius):
    for path_id, path in PATHS.items():
        if intersects_sphere_line(position, radius, path.positions):
            return path_id, path
    return None


class PathFollowingMode(Mode):
    PATH_ID = None
    PARTICLES = None

    def check(self):
        if self.PARTICLES is None or self.PATH_ID is None:
            return

        utilities.notify_all(f"FOLLOWING!")
        START_FOLLOWER(self.PARTICLES, PATHS[self.PATH_ID])
        self.PATH_ID = None
        self.PARTICLES = None

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        self.PARTICLES = [int(index) for index in interaction.particles]
        utilities.notify_all(f"SELECTED PARTICLES: {self.PARTICLES}")
        self.check()

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button != "primary":
            return

        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_paths(next_pos, 1)

        if hovered is not None:
            path_id, path = hovered
            utilities.notify_all(f"SELECTED PATH {path_id}")
            self.PATH_ID = path_id
            SELECTION_OBJECTS.update_line(f"selected.path", positions=path.positions, size=0.15, color=CHOSEN_COLOR)
            self.check()

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_paths(next_pos, 1)

        if hovered is None:
            SELECTION_OBJECTS.remove_line(f"hover.{key}")
        else:
            SELECTION_OBJECTS.update_line(f"hover.{key}", positions=hovered[1].positions, size=0.1, color=HOVER_COLOR)

utilities.add_interaction_mode(PathFollowingMode, "follow")